# 4k Video Upscaler Colab (Real-ESRGAN)
Khang Edited:https://github.com/tmkhang97/4k_Video_Upscaler_Colab_-Real_ESRGAN-

Adapted from: [Real-ESRGAN](https://github.com/xinntao/Real-ESRGAN)

Made with ❤️ by: [yuvraj108c](https://github.com/yuvraj108c)

Github repository: https://github.com/yuvraj108c/4k-video-upscaler-colab

# 1. Setup (~1 minute)

In [13]:
import torch
assert torch.cuda.is_available(), "GPU not detected.. Please change runtime to GPU"

from PIL import Image
import cv2, os, subprocess
from tqdm import tqdm

# Clone Real-ESRGAN nếu chưa có
if not os.path.exists("/content/Real-ESRGAN"):
  !git clone https://github.com/xinntao/Real-ESRGAN.git

%cd /content/Real-ESRGAN

# Cài đặt các thư viện bổ trợ cơ bản trước
!pip install -q gfpgan facexlib ffmpeg ffmpeg-python
!pip install -q addict yapf lmdb tb-nightly

# Khắc phục lỗi build basicsr triệt để bằng cách tải và ép buộc cài đặt bản pre-built wheel không cần compile
!pip install -q https://github.com/dbolya/timm/files/11516752/basicsr-1.4.2-py3-none-any.whl --no-deps

# Cài đặt requirements của Real-ESRGAN và cài đặt dự án ở chế độ không build CUDA
!pip install -q -r requirements.txt
!pip install -q --no-deps -e .

# Tải sẵn file weights của model
!mkdir -p weights
!wget -q --show-progress https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P weights/
!wget -q --show-progress https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-animevideov3.pth -P weights/
!wget -q --show-progress https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth -P weights/

# Hạ cấp numpy để tương thích với Real-ESRGAN
!pip install -q "numpy<2"
mount_drive = False

/content
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
✅ Đã tạo file basicsr/version.py
✅ Đã vá lỗi functional_tensor

🎉 THÀNH CÔNG! Thư viện đã nạp đầy đủ và sẵn sàng xử lý ảnh.



# 2. Mount drive (optional)

In [18]:
from google.colab import drive
mount_drive=False #@param{type:"boolean"}

if mount_drive:
  drive.mount('/content/gdrive/')

MessageError: Error: credential propagation was unsuccessful

# 3. Upscale video

- The upscaled video will be saved to `output_dir`
- If google drive is mounted, it will be also saved at `MyDrive/Upscaled Videos (REAL-ESRGAN)`


In [17]:
import os
import cv2
import subprocess

# 1. Đảm bảo đứng đúng thư mục mã nguồn
%cd /content/Real-ESRGAN

# 2. Tải sẵn weights nếu chưa có
!mkdir -p /content/Real-ESRGAN/weights
!wget -nc -q --show-progress https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-animevideov3.pth -P /content/Real-ESRGAN/weights/
!wget -nc -q --show-progress https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P /content/Real-ESRGAN/weights/

video_path = "/content/cfs2026.mp4" #@param {type:"string"}
output_dir = "/content/output_videos" #@param {type:"string"}
resolution = "4k (3840 x 2160)" # @param ["FHD (1920 x 1080)", "2k (2560 x 1440)", "4k (3840 x 2160)", "2 x original", "3 x original", "4 x original"] {type:"string"}
model = "realesr-animevideov3" #@param ["RealESRGAN_x4plus", "RealESRGAN_x4plus_anime_6B", "realesr-animevideov3"]

os.makedirs(output_dir, exist_ok=True)
assert os.path.exists(video_path), f"Video file does not exist at {video_path}"

video_capture = cv2.VideoCapture(video_path)
video_width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
video_capture.release()

aspect_ratio = float(video_width / video_height)

# Xác định kích thước đích
match resolution:
    case "FHD (1920 x 1080)":
        final_width, final_height = 1920, 1080
    case "2k (2560 x 1440)":
        final_width, final_height = 2560, 1440
    case "4k (3840 x 2160)":
        final_width, final_height = 3840, 2160
    case "2 x original":
        final_width, final_height = 2 * video_width, 2 * video_height
    case "3 x original":
        final_width, final_height = 3 * video_width, 3 * video_height
    case "4 x original":
        final_width, final_height = 4 * video_width, 4 * video_height

if aspect_ratio == 1.0 and "original" not in resolution:
    final_height = final_width
elif aspect_ratio < 1.0 and "original" not in resolution:
    final_width, final_height = final_height, final_width

scale_factor = round(max(final_width / video_width, final_height / video_height), 2)

print(f"Upscaling from {video_width}x{video_height} to {final_width}x{final_height} (scale: {scale_factor})")

# 3. Chạy inference với PYTHONPATH chứa BasicSR và Real-ESRGAN
!PYTHONPATH="/content/BasicSR:/content/Real-ESRGAN" python inference_realesrgan_video.py \
    -n {model} \
    -i '{video_path}' \
    -o '{output_dir}' \
    --outscale {scale_factor}

# Xác định tên file đầu ra của Real-ESRGAN
video_name_with_ext = os.path.basename(video_path)
video_name, _ = os.path.splitext(video_name_with_ext)
upscaled_video_path = os.path.join(output_dir, f"{video_name}_out.mp4")
final_video_name = f"{video_name}_upscaled_{final_width}_{final_height}.mp4"
final_video_path = os.path.join(output_dir, final_video_name)

# 4. Hậu kỳ & crop kích thước chuẩn
if os.path.exists(upscaled_video_path):
    if "original" not in resolution:
        print("Cropping video to exact target aspect ratio...")
        # Sử dụng CPU encoding ổn định, tương thích hoàn toàn trên Colab
        command = f"ffmpeg -loglevel error -y -i '{upscaled_video_path}' -filter:v 'crop={final_width}:{final_height}:(in_w-{final_width})/2:(in_h-{final_height})/2' -c:v libx264 -pix_fmt yuv420p '{final_video_path}'"
        subprocess.run(command, shell=True)
    else:
        command = f"cp '{upscaled_video_path}' '{final_video_path}'"
        subprocess.run(command, shell=True)

    print(f"✅ Video đã xử lý thành công: {final_video_path}")

    # Dọn dẹp file tạm trung gian
    if os.path.exists(upscaled_video_path):
        os.remove(upscaled_video_path)
else:
    print("❌ Lỗi: Quá trình upscale không tạo ra file đầu ra. Vui lòng kiểm tra lại log chi tiết phía trên.")

/content/Real-ESRGAN
Upscaling from 1280x720 to 3840x2160 (scale: 3.0)
inference: 100% 240/240 [06:10<00:00,  1.55s/frame]
Cropping video to exact target aspect ratio...
✅ Video đã xử lý thành công: /content/output_videos/cfs2026_upscaled_3840_2160.mp4


# 4. Disconnect runtime

In [5]:
from google.colab import runtime

disconnect_when_finish = False  #@param{type:"boolean"}

if disconnect_when_finish:
  runtime.unassign()